# Task 3 — Agentic financial research

Paste the research prompt in the input cell. Replace the issuer with a name or ticker (`apple`, `AAPL`, `nvidia`):

> Analyse the current financial health and market sentiment of **[TICKER]**. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.

The notebook extracts the issuer, resolves it, then Agent A chooses tools and returns: financial health, sentiment, three 90-day risks, and one data-driven hedge.

Keys: local `.env`, Colab Secrets. Never paste a key into a cell.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

REPO_URL = os.environ.get(
    "AGENTIC_WORKFLOW_REPO",
    "https://github.com/lavanblavan/Agentic_financial_Analyser.git",
)

def ensure_project() -> Path:
    if IN_COLAB:
        root = Path("/content/Agentic_financial_Analyser")
        if not (root / "src/config.py").exists():
            subprocess.run(["git", "clone", REPO_URL, str(root)], check=True)
        else:
            subprocess.run(["git", "-C", str(root), "pull", "--ff-only"], check=False)
        os.chdir(root)
    else:
        root = Path.cwd().resolve()
        if not (root / "src/config.py").exists():
            raise FileNotFoundError("Open this notebook from the Agentic_financial_Analyser repo root.")

    req = root / "requirements.txt"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    return root

ROOT = ensure_project()
print("project root:", ROOT)
print("runtime:", "colab" if IN_COLAB else "local")

In [ ]:
from src.config import describe_env, load_settings, missing_key_help

settings = load_settings()
print(describe_env(settings))
if not settings.llm_ready:
    print(missing_key_help(settings))

In [ ]:
# Paste the full prompt. Put a name or ticker where [TICKER] is.
# Examples: apple, AAPL, nvidia, tesla
QUERY = """
Analyse the current financial health and market sentiment of apple.
Identify the top three risks to its share price over the next 90 days
and suggest one data-driven hedge strategy.
"""

from src.ticker import parse_research_query

parsed = parse_research_query(QUERY)
COMPANY = parsed["ticker"]
print("task:", parsed["task"])
print(f"{parsed['subject']!r} → {parsed['ticker']}  ({parsed['name']}, via {parsed['resolved_via']})")

## Five tools

Direct invoke on the resolved ticker (no agent). Confirms Task 1 modules still work.

In [ ]:
from src.tools import ALL_TOOLS, get_price_data, calculate_volatility

print("tools:", [t.name for t in ALL_TOOLS])
print(get_price_data.invoke({"ticker": COMPANY})[:500])
print(calculate_volatility.invoke({"ticker": COMPANY, "window_days": 30}))

## Agent A graph

`START → agent → (should_continue) → tools → agent → … → END`

Default model is `openai/gpt-oss-20b` with `max_tokens=800` so Groq free OTPM (1000) is not exceeded. Optional Colab Secret / `.env`: `OPENROUTER_API_KEY` — used automatically if Groq returns 429.

In [ ]:
from src.agent_a import build_agent_a

agent_a = build_agent_a()
print(agent_a.get_graph().draw_mermaid())

## Run Agent A

Uses `QUERY`. Watch `tool_calls`: that list is the LLM’s own order, not a pipeline. The next cell is the answer: health, sentiment, three 90-day risks, one hedge.

In [ ]:
from src.agent_a import format_answer, run_agent_a
from src.config import load_settings

cfg = load_settings()
print("agent model:", cfg.groq_agent_model)
print("max_tokens:", cfg.max_tokens)
print("openrouter:", "yes" if cfg.openrouter_api_key else "no")
result = run_agent_a(QUERY)

print("input:", result["query"].strip())
print("resolved:", result["ticker"], result["parsed"]["name"])
print("tool call order (LLM chose this):")
for i, step in enumerate(result["tool_calls"], start=1):
    print(f"  {i}. {step['tool']}  {step['args']}")
if result.get("missing_after_run"):
    print("still missing:", result["missing_after_run"])

In [ ]:
print(format_answer(result))
print("\n--- structured DataBrief ---")
if result["brief"]:
    from pprint import pprint
    pprint(result["brief"], sort_dicts=False)
else:
    print(result["final_text"])

## Trace log

Each tool write to `logs/agent_trace.jsonl` (name, inputs, truncated output, duration).

In [ ]:
import pandas as pd
from src.tracing import read_trace

trace = read_trace(limit=20)
frame = pd.DataFrame(trace)
cols = [c for c in ["ts", "tool", "ok", "duration_ms", "inputs", "output"] if c in frame.columns]
display(frame[cols] if cols else frame)